###Hash Partitioning

1.It is a default partitioning method in pyspark.

2.It works by assigning a unique hash value to each record based on a  special column and then placing the record in the corresponding partition.

3.This ensures that record with the same value for a specified column are placed in the same partition.

In [51]:
#Importing Required Library

from pyspark.sql import SparkSession

In [52]:
spark= SparkSession.builder.appName("Data Partition").getOrCreate()

In [53]:
df=spark.createDataFrame([
    (1,"Divya",23),
    (2,"Gourav",26),
    (3,"Soham",20),
    (4,"Divya",21),
    (5,"Rohan",22),
],["id","name","age"])

In [54]:
df.show()

+---+------+---+
| id|  name|age|
+---+------+---+
|  1| Divya| 23|
|  2|Gourav| 26|
|  3| Soham| 20|
|  4| Divya| 21|
|  5| Rohan| 22|
+---+------+---+



In [55]:

# (number of partition, column where you want to partition)
df.repartition(4,"id")

DataFrame[id: bigint, name: string, age: bigint]

In [56]:
df.show()

+---+------+---+
| id|  name|age|
+---+------+---+
|  1| Divya| 23|
|  2|Gourav| 26|
|  3| Soham| 20|
|  4| Divya| 21|
|  5| Rohan| 22|
+---+------+---+



In [57]:
# Resilliant Distribution  dataset
# RDD used to  represent an immutable , falult tolerant collectio of elements partitiones accross the cluster node.
# glom() is an RDD Transformation that bundles all individual elements with each separate partition into a single list.
# glom() Used to make bundle of the data

print(df.rdd.glom().collect())

[[Row(id=1, name='Divya', age=23), Row(id=2, name='Gourav', age=26)], [Row(id=3, name='Soham', age=20), Row(id=4, name='Divya', age=21), Row(id=5, name='Rohan', age=22)]]


In [58]:
df.show()

+---+------+---+
| id|  name|age|
+---+------+---+
|  1| Divya| 23|
|  2|Gourav| 26|
|  3| Soham| 20|
|  4| Divya| 21|
|  5| Rohan| 22|
+---+------+---+



### Range Partition


WE divide the whole data into smaller partition based on range of values for a specific column

In [59]:
data=spark.createDataFrame([
    (1,"Divya",23),
    (2,"Gourav",26),
    (3,"Soham",20),
    (4,"Divya",21),
    (5,"Rohan",22),
],["id","name","age"])

In [60]:
data=data.repartitionByRange(3,"age")

In [61]:
print(data.rdd.glom().collect())

[[Row(id=3, name='Soham', age=20), Row(id=4, name='Divya', age=21)], [Row(id=1, name='Divya', age=23), Row(id=5, name='Rohan', age=22)], [Row(id=2, name='Gourav', age=26)]]


###Partitioning Using .partitionBy method


In [62]:
from pyspark.sql import SparkSession
from pyspark.context import SparkContext

In [63]:
spark=SparkSession.builder.appName("sparkDF").getOrCreate()

In [64]:
data_frame=spark.read.csv("/content/products.csv", header=True,inferSchema=True)

In [71]:
data_frame2=spark.read.option("header",True).csv("/content/NetflixOriginals (1).csv",inferSchema=True)

In [65]:
data_frame.show(5)

+---+--------------------+---------------+--------+------+
| id|                name|       category|quantity| price|
+---+--------------------+---------------+--------+------+
|  1|           iPhone 12|    Electronics|      10|899.99|
|  2|     Nike Air Max 90|       Clothing|      25|119.99|
|  3|KitchenAid Stand ...|Home Appliances|       5|299.99|
|  4|    The Great Gatsby|          Books|      50| 12.99|
|  5|L'Oreal Paris Mas...|         Beauty|     100|  9.99|
+---+--------------------+---------------+--------+------+
only showing top 5 rows


In [66]:
data_frame.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



In [68]:
data_frame.write.partitionBy("name","category").mode("overwrite").csv("nameOncategory")

In [69]:
data_frame.show()

+---+--------------------+---------------+--------+------+
| id|                name|       category|quantity| price|
+---+--------------------+---------------+--------+------+
|  1|           iPhone 12|    Electronics|      10|899.99|
|  2|     Nike Air Max 90|       Clothing|      25|119.99|
|  3|KitchenAid Stand ...|Home Appliances|       5|299.99|
|  4|    The Great Gatsby|          Books|      50| 12.99|
|  5|L'Oreal Paris Mas...|         Beauty|     100|  9.99|
|  6|            Yoga Mat|         Sports|      30| 29.99|
|  7| Samsung 4K Smart TV|    Electronics|       8|799.99|
|  8|        Levi's Jeans|       Clothing|      15| 49.99|
|  9|Dyson Vacuum Cleaner|Home Appliances|       3|399.99|
| 10| Harry Potter Series|          Books|      20| 15.99|
| 11|        MAC Lipstick|         Beauty|      75| 16.99|
| 12|Adidas Running Shoes|         Sports|      22| 59.99|
| 13|       PlayStation 5|    Electronics|      12|499.99|
| 14|   Hooded Sweatshirt|       Clothing|      10| 34.9

In [72]:
data_frame2.show()

+--------------------+--------------------+------------------+-------+----------+----------------+
|               Title|               Genre|          Premiere|Runtime|IMDB Score|        Language|
+--------------------+--------------------+------------------+-------+----------+----------------+
|     Enter the Anime|         Documentary|    August 5, 2019|     58|       2.5|English/Japanese|
|         Dark Forces|            Thriller|   August 21, 2020|     81|       2.6|         Spanish|
|             The App|Science fiction/D...| December 26, 2019|     79|       2.6|         Italian|
|      The Open House|     Horror thriller|  January 19, 2018|     94|       3.2|         English|
|         Kaali Khuhi|             Mystery|  October 30, 2020|     90|       3.4|           Hindi|
|               Drive|              Action|  November 1, 2019|    147|       3.5|           Hindi|
|   Leyla Everlasting|              Comedy|  December 4, 2020|    112|       3.7|         Turkish|
|The Last 

###Hash Partition for Netflix data

In [73]:
data_frame2.repartition(4,"Runtime")

DataFrame[Title: string, Genre: string, Premiere: string, Runtime: int, IMDB Score: double, Language: string]

In [74]:
data_frame2.show(5)

+---------------+--------------------+-----------------+-------+----------+----------------+
|          Title|               Genre|         Premiere|Runtime|IMDB Score|        Language|
+---------------+--------------------+-----------------+-------+----------+----------------+
|Enter the Anime|         Documentary|   August 5, 2019|     58|       2.5|English/Japanese|
|    Dark Forces|            Thriller|  August 21, 2020|     81|       2.6|         Spanish|
|        The App|Science fiction/D...|December 26, 2019|     79|       2.6|         Italian|
| The Open House|     Horror thriller| January 19, 2018|     94|       3.2|         English|
|    Kaali Khuhi|             Mystery| October 30, 2020|     90|       3.4|           Hindi|
+---------------+--------------------+-----------------+-------+----------+----------------+
only showing top 5 rows


###Range Partition

In [ ]:
data_frame2.repartitionByRange()